# Avance 1 - Proyecto para el parcial 2 - Contabilidad de Seguros

---

En este avance se realiza lo siguiente:

1. Se define un formato para los archivos .csv de entrada.
2. Se definen funciones de lectura y procesamiento de datos.
    - Se leeran los archivos .csv y se guardaran como un dataframe.
    - Se define un diccionario con los nombres y los valores de las cuentas a usar para calcular las razones financieras.
3. Se definen funciones para calcular las razones financieras.

Como complemento, se desarrollara un script que muestre una GUI desarrollada con tkinter. Dicho script mostrara dos ventanas, la de ingesta de archivos, y la de impresion de resultados. Debido a la naturaleza de tkinter, no se puede incluir dicho codigo en esta libreta de ipython.

## Librerias

---

Primero leemos las librerias a utilizar:

In [1]:
import pandas as pd
import numpy as np 
import sys
import re

## Lectura de datos y el formato de los archivos .csv

Como se indico en el avance 1 del proyecto, nuestro programa recibira como input los archivos .csv que incluyan los dos estados financieros (Balance General y Estado de Resultados) necesarios para el calculo de las razones financieras. Como este tipo de archivo no puede contener multiples hojas, el ususario debera ingresar un .csv en cuya unica hoja este el Balance General, y un .csv cuya unica hoja sea el Estado de Resultados.

Planeamos implementar validaciones y transformaciones, permitiendo que una variedad de formatos (como estan "acomodados" los datos en el .csv) sean ingresados por el usuario; pero para este primer avance, definiremos un formato en especifico para facilitarnos el trabajo.

### El formato a utilizar

Por el momento, el formato de presentacion de los estados de resultados sera el utilizado en la practica de razones financieras. Esto es, crearemos este primer avance asumiendo que los archivos de entrada tienen los mismos nombres de cuentas, con la misma separacion por columnas, con la misma convencion de no tener celdas combinadas, etc.


## Balance general: Lo previo al desarrollo de las funciones; paso a paso.

Ya que definimos el formato de los archivos de entrada, ahora describimos el proceso para el desarrollo del codigo que nos servira para desarrollar nuestras funciones. No sin antes, definir las restricciones (temporales) y los objetivos presentes.


### Restricciones:

> - La fila superior debe de contener una celda que diga "Activo" y una que diga "Pasivo"
> - En la misma fila con nombre de columna "Pasivo", debe haber alguna celda que diga "Capital"

### Objetivos:

> 1. Hacerlo dinamico y robusto para cualquier numero de columnas (en este caso, el numero de columnas en la parte de activos es de 8, incluyendo a la columna "Activos")
> 2. Al hacer transformaciones, no alterar el row_index del dataframe; esto para poder despues implementar una opcion para que el usuario pueda cambiar valores de cuentas clave
> 3. Como cada cuenta solo esta asociada a un valor numerico; obtener un diccionario con los nombres de las cuentas y los valores registrados
>      - Este diccionario se utilizara para el calculo de las razones financieras, y este es el que el usuario alterara (si realiza un cambio en los datos), y por tanto servira para actualizar el dataframe original y actualizar el .csv.
>      - Este diccionario tambien servira para hacer validaciones (concordancia entre estados financieros)

In [4]:
df = pd.read_csv("../INPUT/Balance_general.csv", encoding="utf-8")
display(df)
display(df.columns)

,Activo,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Pasivo,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,NaN,Inversiones,NaN,NaN,NaN,NaN,"7,947,831,726.56",NaN,NaN,Reservas Técnicas,NaN,NaN,NaN,NaN,NaN,NaN,"23,430,131,469.14"
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,De Riesgos en Curso,NaN,NaN,NaN,NaN,"14,191,242,017.45",NaN
2,NaN,NaN,Valores y Operaciones con Productos Derivados,NaN,NaN,"7,488,580,135.83",NaN,NaN,NaN,NaN,NaN,Seguros de Vida,NaN,NaN,NaN,"333,058,132.67",NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Seguros de Accidentes y Enfermedades,NaN,NaN,NaN,"229,078,399.28",NaN
4,NaN,NaN,Valores,NaN,NaN,"7,488,580,135.83",NaN,NaN,NaN,NaN,NaN,Seguros de Daños,NaN,NaN,NaN,"13,629,105,485.50",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Participación No Controladora,NaN,NaN,NaN,NaN,0.00,NaN
75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Suma del Capital,NaN,NaN,NaN,NaN,NaN,NaN,"5,405,180,797.83"
77,NaN,Suma del Activo,NaN,NaN,NaN,NaN,"$39,345,986,020.83",NaN,NaN,Suma del Pasivo y Capital,NaN,NaN,NaN,NaN,NaN,NaN,"39,345,986,020.83"


Index(['Activo', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Pasivo', 'Unnamed: 9',
       'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13',
       'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16'],
      dtype='str')

Podemos ver que el column index del dataframe solo tiene dos valores relevantes: "Activo" y "Pasivo". Estos dos valores nos ayudan a entender la estructura del archivo, pues un balance general tiene la forma:


In [ ]:
# | Activos | Pasivos |
# |         |    +    |
# |         | Capital |

Por lo tanto, usaremos dichos valores para dividir el dataframe en dos: El dataframe asociado a los activos, y el dataframe asociado al capital. Esto lo hacemos para trabajar con dataframes mas simples, es posible que no hagamos esto en las funciones que definamos mas adelante.

### Enfocandonos en el 3er objetivo

En esta seccion desarrollamos snippets de codigo de python para atacar el 3er objetivo. Para no perder el rumbo, a continuacion planteamos objetivos/metas:

- 

Nota extra: break

In [ ]:
# Primer debemos definir la lista de los nombres de columnas asociados a la parte de los activos. Haremos esto dinamico:

new_column_index = [str(column_name).replace("Unnamed: ", "") for column_name in df.columns.tolist()]
new_df = df.copy()
new_df.columns = new_column_index

columnas_activos = [column_name for column_name in new_df.columns.tolist() if (new_df.columns.tolist().index(column_name) < new_df.columns.tolist().index("Pasivo"))]
new_df = new_df.loc[:, columnas_activos]

print(columnas_activos)

display(new_df)

['Activo', '1', '2', '3', '4', '5', '6', '7']


,Activo,1,2,3,4,5,6,7
0,NaN,Inversiones,NaN,NaN,NaN,NaN,"7,947,831,726.56",NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Valores y Operaciones con Productos Derivados,NaN,NaN,"7,488,580,135.83",NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Valores,NaN,NaN,"7,488,580,135.83",NaN,NaN
...,...,...,...,...,...,...,...,...
74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,NaN,Suma del Activo,NaN,NaN,NaN,NaN,"$39,345,986,020.83",NaN


In [11]:
#Balance general
def procesar_bg_csv(ruta_csv: str) -> dict:
    df = pd.read_csv(ruta_csv, encoding="utf-8")

    resultado = {}

    # A (Activos) -> B (columna sin nombre)
    dict_activos = dict(zip(df['Activo'], df['Unnamed: 1']))

    # C (Pasivos) -> D (columna sin nombre)
    dict_pasivos = dict(zip(df['Pasivo'], df['Unnamed: 3']))

    resultado.update(dict_activos)
    resultado.update(dict_pasivos)

    return resultado

#Estado de Resultados
def procesar_er_csv(ruta_csv: str) -> dict:
    df = pd.read_csv(ruta_csv, encoding="utf-8")

    # A (columna sin nombre) -> B (columna sin nombre)
    return dict(zip(df['Unnamed: 0'], df['Unnamed: 1']))

In [12]:
display(procesar_bg_csv("../INPUT/Balance_general.csv"))

{nan: 'Inversiones',
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: 'Inversiones para Obligaciones Laborales',
 nan: nan,
 nan: nan,
 nan: 'Disponibilidad',
 nan: nan,
 nan: nan,
 nan: nan,
 nan: 'Deudores',
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: 'Reaseguradores y Reafianzadores (Neto)',
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: 'Inversiones Permanentes',
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: 'Otros Activos',
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: nan,
 nan: 'Suma de

### Funciones de lectura de datos

Ahora definimos funciones de lectura y procesamiento de datos. Estas funciones estaran contenidas en el modulo de lectura, procesamiento, validacion y transformacion de datos. Recibiran como input las cadenas de texto que contengan la direccion absoluta o relativa de los archivos .csv a leer.